# OceanBase数据库性能智能诊断与优化 - 数据探索

本notebook用于探索OceanBase数据库性能数据和初步分析。

## 目录

1. 数据连接与采集
2. 性能数据可视化
3. 参数相关性分析
4. 特征选择
5. 基准测试分析

In [ ]:
# 导入必要的库
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 添加src目录到路径
project_root = Path().parent.parent
sys.path.insert(0, str(project_root / 'src'))

# 设置可视化样式
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# 配置日志
from loguru import logger
logger.add(sys.stdout, level="INFO")

print("环境配置完成")

## 1. 数据连接与采集

In [ ]:
# 导入数据库连接器
from utils.db_conn import DBConnector
from collector.ob_collector import OBCollector

# 连接数据库（需要先配置 config/db_config.yaml）
try:
    db = DBConnector()
    db.connect()
    print("数据库连接成功")
    
    # 创建采集器
    collector = OBCollector(db)
    
except Exception as e:
    print(f"连接失败: {e}")
    print("请确保已正确配置 config/db_config.yaml")

## 2. 性能数据采集

In [ ]:
# 采集性能数据
metrics = collector.collect_performance_metrics()

for key, df in metrics.items():
    if df is not None:
        print(f"{key}: {len(df)} rows")

## 3. CPU使用率分析

In [ ]:
if metrics.get('cpu_usage') is not None:
    cpu_df = metrics['cpu_usage']
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # CPU使用率分布
    axes[0, 0].hist(cpu_df['cpu_usage'], bins=30, edgecolor='black')
    axes[0, 0].set_title('CPU使用率分布')
    axes[0, 0].set_xlabel('CPU使用率 (%)')
    axes[0, 0].set_ylabel('频数')
    
    # IOPS vs CPU
    axes[0, 1].scatter(cpu_df['cpu_usage'], cpu_df['iops'], alpha=0.6)
    axes[0, 1].set_title('IOPS vs CPU使用率')
    axes[0, 1].set_xlabel('CPU使用率 (%)')
    axes[0, 1].set_ylabel('IOPS')
    
    # 服务器概览
    server_summary = cpu_df.groupby(['svr_ip', 'svr_port'])['cpu_usage'].agg(['mean', 'max', 'std'])
    server_summary.plot(kind='bar', ax=axes[1, 0])
    axes[1, 0].set_title('各服务器CPU概览')
    axes[1, 0].set_xticklabels([f"{r[0]}:{r[1]}" for r in server_summary.index], rotation=45)
    axes[1, 0].legend(['平均', '最大', '标准差'])
    
    # 相关系数热力图
    numeric_cols = cpu_df.select_dtypes(include=[np.number]).columns
    corr_matrix = cpu_df[numeric_cols].corr()
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, ax=axes[1, 1])
    axes[1, 1].set_title('CPU指标相关矩阵')
    
    plt.tight_layout()
    plt.show()

## 4. 慢查询分析

In [ ]:
if metrics.get('slow_queries') is not None:
    slow_df = metrics['slow_queries']
    
    print(f"慢查询总数: {len(slow_df)}")
    print(f"\n慢查询统计:")
    print(slow_df['elapsed_time'].describe())
    
    # 可视化慢查询延迟分布
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 延迟分布
    axes[0].hist(slow_df['elapsed_time'] / 1000, bins=50, edgecolor='black')
    axes[0].set_title('慢查询延迟分布')
    axes[0].set_xlabel('延迟 (秒)')
    axes[0].set_ylabel('查询数量')
    
    # 延迟箱线图
    axes[1].boxplot(slow_df['elapsed_time'] / 1000)
    axes[1].set_title('慢查询延迟箱线图')
    axes[1].set_ylabel('延迟 (秒)')
    
    plt.tight_layout()
    plt.show()

## 5. 参数加载与特征选择

In [ ]:
from b1_analysis.param_loader import ParamLoader
from b1_analysis.feature_selector import FeatureSelector

# 加载参数定义
param_loader = ParamLoader()
print(f"已加载 {len(param_loader)} 个参数")

# 按类别统计参数数量
categories = {}
for param in param_loader.parameters.values():
    cat = param.category
    categories[cat] = categories.get(cat, 0) + 1

print("\n按类别统计:")
for cat, count in categories.items():
    print(f"  {cat}: {count}")

## 6. 模拟容量规划数据

In [ ]:
from b3_capacity.data_simulator import ResourceDataSimulator

# 创建模拟器
simulator = ResourceDataSimulator()

# 生成模拟数据
data = simulator.generate()

print(f"生成数据点: {len(data)}")
print(f"时间范围: {data['timestamp'].min()} 到 {data['timestamp'].max()}")
print(f"\n数据统计:")
print(data[['cpu_usage', 'memory_usage', 'io_usage', 'network_usage']].describe())

## 7. 模拟数据可视化

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 资源使用趋势
axes[0, 0].plot(data['timestamp'], data['cpu_usage'], label='CPU')
axes[0, 0].plot(data['timestamp'], data['memory_usage'], label='内存')
axes[0, 0].plot(data['timestamp'], data['io_usage'], label='IO')
axes[0, 0].plot(data['timestamp'], data['network_usage'], label='网络')
axes[0, 0].set_title('资源使用趋势')
axes[0, 0].set_ylabel('使用率 (%)')
axes[0, 0].legend()
axes[0, 0].tick_params(axis='x', rotation=45)

# 热力图
pivot_data = data.pivot_table(values='cpu_usage', index='day_of_week', columns='hour')
sns.heatmap(pivot_data, cmap='YlOrRd', ax=axes[0, 1])
axes[0, 1].set_title('CPU使用热力图 (星期 x 小时)')
axes[0, 1].set_ylabel('星期')
axes[0, 1].set_xlabel('小时')

# 分布对比
axes[1, 0].boxplot([data['cpu_usage'], data['memory_usage'], data['io_usage'], data['network_usage']],
                labels=['CPU', '内存', 'IO', '网络'])
axes[1, 0].set_title('资源使用分布')
axes[1, 0].set_ylabel('使用率 (%)')

# 峰值分布
peak_hours = data.groupby('hour')['cpu_usage'].max()
axes[1, 1].bar(peak_hours.index, peak_hours.values)
axes[1, 1].set_title('各小时CPU峰值')
axes[1, 1].set_xlabel('小时')
axes[1, 1].set_ylabel('CPU使用率 (%)')

plt.tight_layout()
plt.show()

## 8. 特征选择演示

In [ ]:
# 准备特征数据
feature_cols = ['hour', 'day_of_week', 'is_weekend', 'is_business_hour',
               'cpu_memory_ratio', 'io_network_ratio']
target_col = 'cpu_usage'

X = data[feature_cols].fillna(0)
y = data[target_col]

# 创建特征选择器
selector = FeatureSelector(task_type="regression")

# 基于重要性选择特征
selected_features = selector.select_by_importance(X, y, n_features=3, method="xgboost")
print(f"选择的特征: {selected_features}")

# 绘制特征重要性
if selector.feature_importance is not None:
    plt.figure(figsize=(10, 6))
    top_features = selector.feature_importance.head(10)
    plt.barh(top_features['feature'], top_features['importance'])
    plt.xlabel('重要性')
    plt.title('特征重要性 (XGBoost)')
    plt.tight_layout()
    plt.show()

## 9. 容量规划报告生成

In [ ]:
from b3_capacity.report_generator import CapacityReportGenerator

# 创建报告生成器
report_gen = CapacityReportGenerator()

# 生成摘要
summary = report_gen.generate_summary(data)

print(f"分析周期: {summary.period_start} 到 {summary.period_end}")
print(f"\n当前资源使用:")
for metric, value in summary.current_usage.items():
    print(f"  {metric}: {value:.1f}%")

print(f"\n告警数量: {len(summary.alerts)}")
for alert in summary.alerts:
    print(f"  [{alert.severity.upper()}] {alert.metric}: {alert.current_value:.1f}%")
    print(f"    建议: {alert.recommendation}")

## 10. MLflow实验追踪演示

In [ ]:
from b4_validation.experiment_tracker import ExperimentTracker

# 创建实验追踪器
try:
    tracker = ExperimentTracker(
        tracking_uri="./experiments/mlflow",
        experiment_name="oceanbase_tuning_exploration"
    )
    
    # 开始一个实验运行
    run_id = tracker.start_run(
        run_name="exploration_test",
        tags={"mode": "exploration", "user": "data_scientist"}
    )
    
    print(f"实验运行ID: {run_id}")
    
    # 记录参数
    tracker.log_params({
        'model_type': 'xgboost',
        'n_features': len(feature_cols),
        'data_size': len(data)
    })
    
    # 记录指标
    tracker.log_resource_metrics({
        'cpu_usage': summary.current_usage.get('cpu_usage', 0),
        'memory_usage': summary.current_usage.get('memory_usage', 0),
        'io_usage': summary.current_usage.get('io_usage', 0),
        'network_usage': summary.current_usage.get('network_usage', 0)
    })
    
    # 结束运行
    tracker.end_run()
    
    print("实验记录完成")
    
except ImportError:
    print("MLflow未安装，跳过实验追踪演示")

## 总结

本notebook演示了:
1. 数据库连接与性能数据采集
2. CPU、内存、IO等资源使用情况分析
3. 慢查询分析
4. 参数定义加载
5. 模拟数据生成与可视化
6. 特征选择
7. 容量规划报告生成
8. MLflow实验追踪